In [1]:
# Import library
import torch
import torchaudio
import librosa
import numpy as np
import soundfile as sf
from pydub import AudioSegment
import random

c:\SEM 5\LAB AI Semester 5\.venv\Lib\site-packages\pydub\utils.py:170: RuntimeWarning: Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work
  warn("Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work", RuntimeWarning)


In [2]:
# Load Data

bundle = torchaudio.pipelines.WAV2VEC2_ASR_BASE_960H
model = bundle.get_model()

audio_file = 'sample_audio.wav'
audio, sr = librosa.load(audio_file, sr=None)
clip = AudioSegment.from_wav(audio_file)

In [ ]:
# Export Helper Func

def export_pydub(audio):
    while True:
        file_name = input("input filename: ")
        if file_name.endswith(".wav"):
            break
        else:
            print("must be end with .wav")
    audio.export(file_name, format="wav")
    print("Audio filename saved as", file_name)
    
def export_librosa(audio):
    while True:
        file_name = input("input filename: ")
        if file_name.endswith(".wav"):
            break
        else:
            print("must be end with .wav")
    sf.write(file_name,np.clip(audio, -1, 1), samplerate=sr)
    print("Audio filename saved as", file_name)

In [4]:
# Predict Text
def predict_text(audio_file):
    waveform, sr = torchaudio.load(audio_file)
    # waveform = waveform.mean(dim=0, keepdim=True) # stereo -> mono
    
    if sr != bundle.sample_rate:
        waveform = torchaudio.functional.resample(waveform, sr, bundle.sample_rate)
        
    with torch.inference_mode():
        emission, _ = model(waveform)
        
    class GreedyCTCDecoder(torch.nn.Module):
        def __init__(self, labels):
            super().__init__()
            self.labels = labels
            self.blank = 0
            
        def forward(self, emission: torch.Tensor) -> str:
            indices = torch.argmax(emission, dim=-1)
            indices = torch.unique_consecutive(indices, dim=-1)
            indices = [i for i in indices if i != self.blank]
            return "".join([self.labels[i] for i in indices])
        
    decoder = GreedyCTCDecoder(bundle.get_labels())
    transcript = decoder(emission[0])
    transcript = transcript.replace("|", " ")
    return transcript

In [5]:
# Augmentation volume, pitch shift, white noise, speed, random gain

def adjvolume(clip):
    while True:
        db = float(input("Input DB for adjust volume (-30 to 30): "))
        if -30 <= db <= 30:
            break
        else:
            print("must be between -30 to 30")
    export_pydub(clip + db)
    
def pitch_shift():
    while True:
        p_rate = float(input("Input pitch rate (-5 to 5): "))
        if -5 <= p_rate <= 5:
            break           
        else:

            print("must be between -5 to 5")
    pitch_result = librosa.effects.pitch_shift(audio, sr=sr, n_steps=p_rate)
    export_librosa(pitch_result)
    
def white_noise():
    while True:
        level = float(input("Input noise level (0.1 to 0.5): "))
        if 0.1 <= level <= 0.5:
            break
        else:
            print("must be between 0.1 to 0.5")

    noise = np.random.normal(0, audio.std(), audio.size)
    export_librosa(audio + noise * level)
    
def adjspeed():
    while True:
        rate = float(input("Input pitch rate (0.5 to 2): "))
        if 0.5 <= rate <= 2:
            break
        else:
            print("must be between 0.5 to 2")
    speed_rate = librosa.effects.time_stretch(audio, rate=rate)
    export_librosa(speed_rate)
    
def randomgain():
    gain = np.random.uniform(0.5, 1.5)
    print("random gain: ", gain)
    export_librosa(audio * gain)

In [6]:
def augmenu():
    while True:
        print("=== Aug Menu ===")
        print("1. Adjust Volume")
        print("2. Adjust Pitch Shift")
        print("3. White Noise")
        print("4. Adjust Speed")
        print("5. Random Gain")
        print("6. Exit")
        
        choice_aug = input(">> ")
        if choice_aug == "1":
            adjvolume(clip)
        elif choice_aug == "2":
            pitch_shift()
        elif choice_aug == "3":
            white_noise()
        elif choice_aug == "4":
            adjspeed()
        elif choice_aug == "5":
            randomgain()
        elif choice_aug == "6":
            print("exit to main menu")
            break
        else:
            print("input must be between 1-6")

In [7]:
def mainmenu():
    while True:
        print("=== Main Menu ===")
        print("1. Audio Augmentation")
        print("2. Predict Text")
        print("3. Exit")
        choice = input(">>")
        
        if choice == "1":
            augmenu()
        elif choice == "2":
            transcript = predict_text(audio_file)
            print("Transcript Results: ")
            print(transcript)
        elif choice == "3":
            print("end")
            break
        else:
            print("must be between 1-3")

mainmenu()

=== Main Menu ===
1. Audio Augmentation
2. Predict Text
3. Exit


RuntimeError: Failed to create AudioDecoder for sample_audio.wav: Could not load libtorchcodec. Likely causes:
          1. FFmpeg is not properly installed in your environment. We support
             versions 4, 5, 6, 7, 8, and 9, and we attempt to load libtorchcodec
             for each of those versions. Errors for versions not installed on
             your system are expected; only the error for your installed FFmpeg
             version is relevant. On Windows, ensure you've installed the
             "full-shared" version which ships DLLs.
          2. The PyTorch version (2.13.0+cpu) is not compatible with
             this version of TorchCodec. Refer to the version compatibility
             table:
             https://github.com/pytorch/torchcodec?tab=readme-ov-file#installing-torchcodec.
          3. Another runtime dependency; see exceptions below.

        The following exceptions were raised as we tried to load libtorchcodec:
        
[start of libtorchcodec loading tr